In [1]:
import json
import pandas as pd
import os
import duckdb
folder_path = "json_files/"

In [44]:
def empty_json():
    for item in os.listdir(folder_path):
        # Get the full path of the item
        full_path = os.path.join(folder_path, item)
        if full_path != 'json_files/key_info.json':
            empty = []
            with open(full_path, 'w') as file:
                json.dump(empty, file, indent=4)
            print('overwritten ', full_path)
        else:
            reset_key = [{
                            "last_pk": 0
                        }]
            with open(full_path, 'w') as file:
                json.dump(reset_key, file, indent=4)
            print('key reset')
            


In [45]:
empty_json()

overwritten  json_files/ISF.json
key reset
overwritten  json_files/location.json
overwritten  json_files/media.json
overwritten  json_files/personnel.json
overwritten  json_files/raw.json
overwritten  json_files/site_activity.json
overwritten  json_files/subsystem.json
overwritten  json_files/systembaseline.json
overwritten  json_files/test.json


In [47]:
with open(folder_path + 'system_baseline.json', 'r') as file:
    systembaseline_list = json.load(file)

In [48]:
with open(folder_path + 'site_activity.json', 'r') as file:
    site_activity_list = json.load(file)

In [49]:
systembaseline_list

[{'FK': 1, 'system_baseline': '4.7.3.2'},
 {'FK': 2, 'system_baseline': '4.8.0'},
 {'FK': 3, 'system_baseline': '4.8.9.3'}]

In [50]:
site_activity_list

[{'Timestamp': '2026-10-14T00:45:16.624000+08:00',
  'Date of Site Activity': '2026-10-14T00:00:00+08:00',
  'Start Time': '08:00',
  'End Time': '18:00',
  'Title': 'Title',
  'Activity Description': 'Testing',
  'Send': 1,
  'PK': 1},
 {'Timestamp': '2026-10-19T01:03:00.914000+08:00',
  'Date of Site Activity': '2026-10-19T00:00:00+08:00',
  'Start Time': '08:00',
  'End Time': '18:00',
  'Title': 'Title Test',
  'Activity Description': 'second test',
  'Send': 1,
  'PK': 2},
 {'Timestamp': '2026-11-04T00:07:15.507000+08:00',
  'Date of Site Activity': '2026-11-04T00:00:00+08:00',
  'Start Time': '08:00',
  'End Time': '18:45',
  'Title': 'ATC Dynamic Day 4',
  'Activity Description': 'Scanned Balises 4 - 10',
  'Send': 1,
  'PK': 3}]

In [51]:
df_site_activity = pd.DataFrame(site_activity_list)

In [52]:
df_site_activity

,Timestamp,Date of Site Activity,Start Time,End Time,Title,Activity Description,Send,PK
0,2026-10-14T00:45:16.624000+08:00,2026-10-14T00:00:00+08:00,08:00,18:00,Title,Testing,1,1
1,2026-10-19T01:03:00.914000+08:00,2026-10-19T00:00:00+08:00,08:00,18:00,Title Test,second test,1,2
2,2026-11-04T00:07:15.507000+08:00,2026-11-04T00:00:00+08:00,08:00,18:45,ATC Dynamic Day 4,Scanned Balises 4 - 10,1,3


In [53]:
new_columns = [df_site_activity.columns[-1]] + list(df_site_activity.columns[:-1])
df_site_activity = df_site_activity[new_columns]

In [54]:
df_site_activity.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   PK                     3 non-null      int64
 1   Timestamp              3 non-null      str  
 2   Date of Site Activity  3 non-null      str  
 3   Start Time             3 non-null      str  
 4   End Time               3 non-null      str  
 5   Title                  3 non-null      str  
 6   Activity Description   3 non-null      str  
 7   Send                   3 non-null      int64
dtypes: int64(2), str(6)
memory usage: 324.0 bytes


In [55]:
all_df = {}

for item in os.listdir(folder_path):
        # Get the full path of the item
    full_path = os.path.join(folder_path, item)
    print(full_path)
    with open(full_path, 'r') as file:
        new_table = json.load(file)
    idx = item.index('.')
    all_df[item[:idx]] = pd.DataFrame(new_table)
    all_df[item[:idx]].columns = all_df[item[:idx]].columns.str.replace(" ", "_")
    new_columns = [all_df[item[:idx]].columns[-1]] + list(all_df[item[:idx]].columns[:-1])
    all_df[item[:idx]] = all_df[item[:idx]][new_columns]

all_df.keys()

json_files/ISF.json
json_files/key_info.json
json_files/location.json
json_files/media.json
json_files/personnel.json
json_files/raw.json
json_files/site_activity.json
json_files/subsystem.json
json_files/system_baseline.json
json_files/test.json


dict_keys(['ISF', 'key_info', 'location', 'media', 'personnel', 'raw', 'site_activity', 'subsystem', 'system_baseline', 'test'])

In [56]:
all_df['raw']['Timestamp'] = pd.to_datetime(all_df['raw']['Timestamp'])
all_df['raw']['Date_of_Site_Activity'] = pd.to_datetime(all_df['raw']['Date_of_Site_Activity'])
all_df['site_activity']['Timestamp'] = pd.to_datetime(all_df['site_activity']['Timestamp'])
all_df['site_activity']['Date_of_Site_Activity'] = pd.to_datetime(all_df['site_activity']['Date_of_Site_Activity'])

In [57]:
locals().update(all_df)

In [58]:
query = """
SELECT team AS Team, name AS Name
FROM personnel
ORDER BY Team
"""
result_df = duckdb.query(query).to_df()
result_df

,Team,Name
0,J152,Erwin
1,J152,Pablo
2,J152,Francisco
3,J152,Joana
4,J152,Keri
5,J152,Jordi
6,J152,Richter
7,JRLP,Jia Ling
8,JRLP,Herman
9,JRLP,Ern-Min


Search by Date

In [59]:
primary_key = 1
from_date = '2026-09-10'
to_date = '2026-12-31'

In [60]:
query = """
SELECT *
FROM site_activity
WHERE 1=1
AND PK = $primary_key
AND Date_of_Site_Activity > $from_date
AND Date_of_Site_Activity < $to_date

"""
result_df = duckdb.query(query, params={"primary_key": primary_key, "from_date": from_date, "to_date": to_date}).to_df()
result_df

,PK,Timestamp,Date_of_Site_Activity,Start_Time,End_Time,Title,Activity_Description,Send
0,1,2026-10-14 00:45:16.624000+08:00,2026-10-14 00:00:00+08:00,08:00,18:00,Title,Testing,1


Show All Test Cases

In [61]:
query = """
SELECT test, COUNT(status) AS attempts, SUM(CASE WHEN status = 'Passed' THEN 1 ELSE 0 END) * 100 / COUNT(status) AS Passing_Rate
FROM test
GROUP BY test
"""
result_df = duckdb.query(query).to_df()
result_df

,test,attempts,Passing_Rate
0,TTR2.6,1,0.0
1,TTR2.11,1,0.0
2,TTR1.3,1,0.0
3,TTR1.4,1,0.0
4,TTR1.2,1,100.0
5,TTR1.6,1,0.0
6,TTR1.1,1,100.0
7,TTR2.3,1,100.0
8,TTR2.5,1,100.0
9,TTR2.7,1,0.0


Find one test case's media

In [62]:
test_case = 'TTR1.2'

In [63]:
query = """
SELECT test, status, Date_of_Site_Activity, Title, Activity_Description, media
FROM site_activity
JOIN test ON test.FK = site_activity.PK
JOIN media ON site_activity.PK = media.FK
WHERE test = $test_case
"""
result_df = duckdb.query(query, params={"test_case": test_case}).to_df()
result_df

,test,status,Date_of_Site_Activity,Title,Activity_Description,media
0,TTR1.2,Passed,2026-10-14 00:00:00+08:00,Title,Testing,https://drive.google.com/open?id=1gSc1Kw7H6slI...
1,TTR1.2,Passed,2026-10-14 00:00:00+08:00,Title,Testing,https://drive.google.com/open?id=1uM6nE2A8g0Pv...
2,TTR1.2,Passed,2026-10-14 00:00:00+08:00,Title,Testing,https://drive.google.com/open?id=1gSc1Kw7H6slI...
3,TTR1.2,Passed,2026-10-14 00:00:00+08:00,Title,Testing,https://drive.google.com/open?id=1uM6nE2A8g0Pv...


Show All Personnel

In [64]:
query = """
SELECT name, team, COUNT(Date_of_Site_Activity) AS site_count
FROM personnel
JOIN site_activity ON site_activity.PK = personnel.FK
GROUP BY name, team
"""
result_df = duckdb.query(query).to_df()
result_df

,name,team,site_count
0,Jia Ling,JRLP,1
1,Herman,JRLP,1
2,Erwin,J152,1
3,Richter,J152,1
4,Jia Wei,JRLP,1
5,Francisco,J152,1
6,Jordi,J152,1
7,Ern-Min,JRLP,1
8,Yidi,JRLP,1
9,Jazli,JRLP,1


In [65]:
query = """
SELECT Date_of_Site_Activity, PK, Start_Time, End_Time, Title, JRLP, J152
FROM raw
"""
result_df = duckdb.query(query).to_df()
result_df

,Date_of_Site_Activity,PK,Start_Time,End_Time,Title,JRLP,J152
0,2026-10-19 00:00:00+08:00,1,08:00,18:00,Title Test,"Yidi, Jazli","Francisco, Joana"
1,2026-10-14 00:00:00+08:00,1,08:00,18:00,Title,"Jia Ling, Herman, Ern-Min","Erwin, Pablo"
2,2026-11-04 00:00:00+08:00,1,08:00,18:45,ATC Dynamic Day 4,Jia Wei,"Keri, Jordi, Richter"
3,2026-10-14 00:00:00+08:00,1,08:00,18:00,Title,"Jia Ling, Herman, Ern-Min","Erwin, Pablo"
4,2026-10-19 00:00:00+08:00,2,08:00,18:00,Title Test,"Yidi, Jazli","Francisco, Joana"
5,2026-11-04 00:00:00+08:00,3,08:00,18:45,ATC Dynamic Day 4,Jia Wei,"Keri, Jordi, Richter"


Show all ISFs/Search for ISF

In [66]:
ISF_no = None
status = None
#None to select all ISFs

In [67]:
query = """
SELECT ISF, Status, PK AS site_id, Date_of_Site_Activity, Title
FROM ISF
JOIN site_activity ON ISF.FK = site_activity.PK
WHERE (ISF.ISF = $ISF_no OR $ISF_no IS NULL)
AND (ISF.status = $status OR $status IS NULL)


"""
result_df = duckdb.query(query, params={"ISF_no": ISF_no, "status": status}).to_df()
result_df

,ISF,status,site_id,Date_of_Site_Activity,Title
0,ISF001,Opened,1,2026-10-14 00:00:00+08:00,Title
1,ISF002,Closed,1,2026-10-14 00:00:00+08:00,Title
2,ISF005,Closed,1,2026-10-14 00:00:00+08:00,Title
3,ISF003,Failed,1,2026-10-14 00:00:00+08:00,Title
4,ISF004,Failed,1,2026-10-14 00:00:00+08:00,Title
5,ISF006,Failed,1,2026-10-14 00:00:00+08:00,Title
6,ISF015,Closed,2,2026-10-19 00:00:00+08:00,Title Test
7,ISF016,Failed,2,2026-10-19 00:00:00+08:00,Title Test


Show all SyBL

In [89]:
sysbl = '4.7.3.2'
#None to select all sysbl

In [80]:
query = """
SELECT system_baseline, PK, Date_of_Site_Activity, Title
FROM site_activity
JOIN system_baseline ON system_baseline.FK = site_activity.PK
WHERE (system_baseline = $sysbl OR $sysbl IS NULL)
"""
result_df = duckdb.query(query, params={"sysbl": sysbl}).to_df()
result_df


,system_baseline,PK,Date_of_Site_Activity,Title
0,4.7.3.2,1,2026-10-14 00:00:00+08:00,Title
1,4.8.0,2,2026-10-19 00:00:00+08:00,Title Test
2,4.8.9.3,3,2026-11-04 00:00:00+08:00,ATC Dynamic Day 4


In [92]:
query = """
SELECT system_baseline, ISF, status, Date_of_Site_Activity, Title
FROM site_activity
JOIN system_baseline ON system_baseline.FK = site_activity.PK
JOIN ISF ON site_activity.PK = ISF.FK
WHERE (system_baseline = $sysbl OR $sysbl IS NULL)
"""
result_df = duckdb.query(query, params={"sysbl": sysbl}).to_df()
result_df


,system_baseline,ISF,status,Date_of_Site_Activity,Title
0,4.7.3.2,ISF001,Opened,2026-10-14 00:00:00+08:00,Title
1,4.7.3.2,ISF002,Closed,2026-10-14 00:00:00+08:00,Title
2,4.7.3.2,ISF005,Closed,2026-10-14 00:00:00+08:00,Title
3,4.7.3.2,ISF003,Failed,2026-10-14 00:00:00+08:00,Title
4,4.7.3.2,ISF004,Failed,2026-10-14 00:00:00+08:00,Title
5,4.7.3.2,ISF006,Failed,2026-10-14 00:00:00+08:00,Title


Search by Location

In [ ]:
location1 = 'JS1'
location2 = 'TGD'
location3 = None
location4 = None
location5 = None
all_locations = True
#put all locations as True or False then the rest of the search does not matter

In [125]:
query = """
SELECT location, PK, Date_of_Site_Activity, Title
FROM site_activity
JOIN location ON site_activity.PK = location.FK
WHERE (location = $location1)
OR (location = $location2)
OR ($all_locations)
"""
result_df = duckdb.query(query, params={"location1": location1, "location2": location2, "all_locations": all_locations}).to_df()
# result_df = duckdb.execute(query, params={"location1": location1, "location2": location2, "location3": location3, "location4": location4, "location5": location5}).to_df()
result_df


,location,PK,Date_of_Site_Activity,Title
0,JS4,1,2026-10-14 00:00:00+08:00,Title
1,JS5,1,2026-10-14 00:00:00+08:00,Title
2,JE6,1,2026-10-14 00:00:00+08:00,Title
3,TGD,2,2026-10-19 00:00:00+08:00,Title Test
4,JS1,2,2026-10-19 00:00:00+08:00,Title Test
5,JS5,2,2026-10-19 00:00:00+08:00,Title Test
6,TGD,3,2026-11-04 00:00:00+08:00,ATC Dynamic Day 4
